# 15 AlphaEarth city inventory
Goal: scan the project and identify which additional cities are ready for the AlphaEarth workflow.

In [1]:
from pathlib import Path
import pandas as pd
import re

# =========================
# paths
# =========================
ROOT = Path("/Users/yufeizhou/Desktop/heat-exposure-compare")
PROCESSED = ROOT / "data_processed"
ALPHA_OUT = ROOT / "outputs" / "pilot" / "alphaearth"
SAVE_DIR = ALPHA_OUT
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("PROCESSED exists:", PROCESSED.exists())
print("ALPHA_OUT exists:", ALPHA_OUT.exists())

# =========================
# helper functions
# =========================
def pick_first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

def first_match(folder: Path, patterns):
    if not folder.exists():
        return None
    for pat in patterns:
        matches = sorted(folder.glob(pat))
        if matches:
            return matches[0]
    return None

def city_from_alpha_file(path: Path):
    """
    Try to parse city name from files like:
    houston_alphaearth_2024_tract_mean_mosaic_clean.csv
    phoenix_alphaearth_pc7_scores_corrected.csv
    """
    stem = path.stem.lower()
    m = re.match(r"^([a-z0-9]+)_alphaearth_", stem)
    if m:
        return m.group(1)
    return None

# =========================
# collect candidate city names
# =========================
city_names = set()

# from data_processed subfolders
if PROCESSED.exists():
    for p in PROCESSED.iterdir():
        if p.is_dir():
            city_names.add(p.name.lower())

# from alphaearth output files
if ALPHA_OUT.exists():
    for p in ALPHA_OUT.glob("*alphaearth*.csv"):
        city = city_from_alpha_file(p)
        if city:
            city_names.add(city)

city_names = sorted(city_names)

print("\nDetected city candidates:")
print(city_names)

# =========================
# build inventory
# =========================
rows = []

for city in city_names:
    city_folder = PROCESSED / city

    # preferred heat file patterns
    heat_patterns = [
        f"{city}_master_with_lst_hi_fixed.gpkg",
        f"{city}_master_with_lst_hi.gpkg",
        f"{city}_master_lst_hi_hotcompare.gpkg",
        f"{city}_master_with_lst.gpkg",
        f"{city}*lst*hi*.gpkg",
        f"{city}*.gpkg",
    ]
    heat_file = first_match(city_folder, heat_patterns)

    # preferred embedding file patterns
    embed_patterns = [
        f"{city}_alphaearth_2024_tract_mean_mosaic_clean.csv",
        f"{city}_alphaearth_*_tract_mean_mosaic_clean.csv",
        f"{city}_alphaearth_*mean_mosaic_clean.csv",
        f"{city}_alphaearth_*mean_mosaic*.csv",
    ]
    embed_file = None
    for pat in embed_patterns:
        matches = sorted(ALPHA_OUT.glob(pat))
        if matches:
            embed_file = matches[0]
            break

    # preferred PC score patterns
    pc_patterns = [
        f"{city}_alphaearth_pc7_scores_corrected.csv",
        f"{city}_alphaearth_pc*_scores_corrected.csv",
        f"{city}_alphaearth_pc*_scores*.csv",
    ]
    pc_file = None
    for pat in pc_patterns:
        matches = sorted(ALPHA_OUT.glob(pat))
        if matches:
            pc_file = matches[0]
            break

    rows.append({
        "city": city,
        "has_heat_file": heat_file is not None,
        "heat_file": str(heat_file) if heat_file else "",
        "has_embed_file": embed_file is not None,
        "embed_file": str(embed_file) if embed_file else "",
        "has_pc_file": pc_file is not None,
        "pc_file": str(pc_file) if pc_file else "",
    })

inventory_df = pd.DataFrame(rows).sort_values(
    ["has_heat_file", "has_embed_file", "has_pc_file", "city"],
    ascending=[False, False, False, True]
).reset_index(drop=True)

# ready definitions
inventory_df["ready_for_full_workflow"] = (
    inventory_df["has_heat_file"] & inventory_df["has_embed_file"] & inventory_df["has_pc_file"]
)

inventory_df["ready_if_we_recompute_pc"] = (
    inventory_df["has_heat_file"] & inventory_df["has_embed_file"]
)

# exclude existing anchor cities
inventory_df["is_anchor_city"] = inventory_df["city"].isin(["houston", "phoenix"])

# priority score for selecting next cities
inventory_df["priority_score"] = (
    inventory_df["ready_for_full_workflow"].astype(int) * 3
    + inventory_df["ready_if_we_recompute_pc"].astype(int) * 2
    + (~inventory_df["is_anchor_city"]).astype(int) * 1
)

inventory_df = inventory_df.sort_values(
    ["priority_score", "ready_for_full_workflow", "ready_if_we_recompute_pc", "city"],
    ascending=[False, False, False, True]
).reset_index(drop=True)

# subsets
ready_full_df = inventory_df[
    (~inventory_df["is_anchor_city"]) & (inventory_df["ready_for_full_workflow"])
].copy()

ready_recompute_df = inventory_df[
    (~inventory_df["is_anchor_city"]) & (inventory_df["ready_if_we_recompute_pc"])
].copy()

missing_df = inventory_df[
    ~(inventory_df["ready_if_we_recompute_pc"])
].copy()

# suggest next 2 cities
suggest_df = ready_recompute_df.head(2).copy()

# =========================
# save
# =========================
inventory_csv = SAVE_DIR / "city_inventory_alphaearth.csv"
ready_full_csv = SAVE_DIR / "city_inventory_ready_full.csv"
ready_recompute_csv = SAVE_DIR / "city_inventory_ready_recompute_pc.csv"
missing_csv = SAVE_DIR / "city_inventory_missing.csv"
suggest_csv = SAVE_DIR / "city_inventory_suggest_next2.csv"

inventory_df.to_csv(inventory_csv, index=False)
ready_full_df.to_csv(ready_full_csv, index=False)
ready_recompute_df.to_csv(ready_recompute_csv, index=False)
missing_df.to_csv(missing_csv, index=False)
suggest_df.to_csv(suggest_csv, index=False)

# =========================
# display
# =========================
print("\n===== Full inventory =====")
display(inventory_df)

print("\n===== Ready for full workflow (new cities only) =====")
display(ready_full_df)

print("\n===== Ready if we recompute PC (new cities only) =====")
display(ready_recompute_df)

print("\n===== Missing / not ready =====")
display(missing_df)

print("\n===== Suggested next 2 cities =====")
display(suggest_df)

print("\nSaved files:")
print(inventory_csv)
print(ready_full_csv)
print(ready_recompute_csv)
print(missing_csv)
print(suggest_csv)

ROOT: /Users/yufeizhou/Desktop/heat-exposure-compare
PROCESSED exists: True
ALPHA_OUT exists: True

Detected city candidates:
['city', 'houston', 'phoenix']

===== Full inventory =====


,city,has_heat_file,heat_file,has_embed_file,embed_file,has_pc_file,pc_file,ready_for_full_workflow,ready_if_we_recompute_pc,is_anchor_city,priority_score
0,houston,True,/Users/yufeizhou/Desktop/heat-exposure-compare...,True,/Users/yufeizhou/Desktop/heat-exposure-compare...,True,/Users/yufeizhou/Desktop/heat-exposure-compare...,True,True,True,5
1,phoenix,True,/Users/yufeizhou/Desktop/heat-exposure-compare...,True,/Users/yufeizhou/Desktop/heat-exposure-compare...,True,/Users/yufeizhou/Desktop/heat-exposure-compare...,True,True,True,5
2,city,False,,False,,False,,False,False,False,1



===== Ready for full workflow (new cities only) =====


,city,has_heat_file,heat_file,has_embed_file,embed_file,has_pc_file,pc_file,ready_for_full_workflow,ready_if_we_recompute_pc,is_anchor_city,priority_score



===== Ready if we recompute PC (new cities only) =====


...


===== Missing / not ready =====


,city,has_heat_file,heat_file,has_embed_file,embed_file,has_pc_file,pc_file,ready_for_full_workflow,ready_if_we_recompute_pc,is_anchor_city,priority_score
2,city,False,,False,,False,,False,False,False,1



===== Suggested next 2 cities =====


,city,has_heat_file,heat_file,has_embed_file,embed_file,has_pc_file,pc_file,ready_for_full_workflow,ready_if_we_recompute_pc,is_anchor_city,priority_score



Saved files:
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/city_inventory_alphaearth.csv
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/city_inventory_ready_full.csv
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/city_inventory_ready_recompute_pc.csv
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/city_inventory_missing.csv
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/city_inventory_suggest_next2.csv


In [2]:
from pathlib import Path
import pandas as pd
import re

# =========================
# paths
# =========================
ROOT = Path("/Users/yufeizhou/Desktop/heat-exposure-compare")
DATA_PROCESSED = ROOT / "data_processed"
CITY_INTAKE = ROOT / "city_intake"
OUT = ROOT / "outputs" / "pilot" / "alphaearth"
OUT.mkdir(parents=True, exist_ok=True)

ANCHOR_CITIES = {"houston", "phoenix"}

print("ROOT:", ROOT)
print("DATA_PROCESSED exists:", DATA_PROCESSED.exists())
print("CITY_INTAKE exists:", CITY_INTAKE.exists())
print("OUT exists:", OUT.exists())

# =========================
# helper functions
# =========================
def clean_city_name(name: str) -> str:
    name = name.lower().strip()
    name = re.sub(r"[^a-z0-9_]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

def score_file_name(name: str) -> int:
    """
    Higher = more likely to be the right city-level heat/input file.
    """
    s = name.lower()
    score = 0

    # reward likely useful files
    if s.endswith(".gpkg"):
        score += 5
    if "master" in s:
        score += 4
    if "lst" in s:
        score += 3
    if "hi" in s:
        score += 3
    if "fixed" in s:
        score += 2
    if "hotcompare" in s:
        score += 2
    if "tract" in s:
        score += 2

    # penalize obviously derived / non-input files
    bad_terms = [
        "alphaearth", "summary", "meeting", "teacher", "compare",
        "delta", "score", "scores", "pc", "png", "md", "txt"
    ]
    for t in bad_terms:
        if t in s:
            score -= 5

    return score

def list_candidate_files(folder: Path):
    if not folder.exists():
        return []

    candidates = []
    for p in folder.rglob("*"):
        if not p.is_file():
            continue

        s = p.name.lower()

        # only keep plausible geospatial/table city inputs
        if p.suffix.lower() not in {".gpkg", ".geojson", ".csv", ".parquet"}:
            continue

        # must look somewhat relevant to tract/heat workflow
        good_signal = any(k in s for k in [
            "lst", "hi", "heat", "master", "tract", "hotcompare"
        ])
        if not good_signal:
            continue

        candidates.append(p)

    return candidates

def infer_city_from_path(path: Path):
    """
    Infer city from parent folder first, then filename.
    """
    # try parent folder
    if path.parent.name not in {"data_processed", "city_intake"}:
        parent_city = clean_city_name(path.parent.name)
        if parent_city:
            return parent_city

    # fallback: first token of filename
    stem = clean_city_name(path.stem)
    if "_" in stem:
        return stem.split("_")[0]
    return stem

# =========================
# scan folders
# =========================
all_candidates = []

for base_folder, source_tag in [
    (DATA_PROCESSED, "data_processed"),
    (CITY_INTAKE, "city_intake")
]:
    files = list_candidate_files(base_folder)
    print(f"\n{source_tag}: found {len(files)} plausible files")

    for p in files:
        city = infer_city_from_path(p)
        row = {
            "city": city,
            "source_tag": source_tag,
            "file_path": str(p),
            "file_name": p.name,
            "suffix": p.suffix.lower(),
            "score": score_file_name(p.name),
            "size_mb": round(p.stat().st_size / (1024 * 1024), 3)
        }
        all_candidates.append(row)

cand_df = pd.DataFrame(all_candidates)

if cand_df.empty:
    print("\nNo candidate files found.")
else:
    cand_df = cand_df.sort_values(
        ["city", "score", "size_mb", "file_name"],
        ascending=[True, False, False, True]
    ).reset_index(drop=True)

display(cand_df.head(50))

# =========================
# choose best file per city
# =========================
if cand_df.empty:
    best_df = pd.DataFrame(columns=[
        "city", "source_tag", "file_path", "file_name", "suffix", "score", "size_mb"
    ])
else:
    best_df = (
        cand_df.sort_values(
            ["city", "score", "size_mb", "file_name"],
            ascending=[True, False, False, True]
        )
        .groupby("city", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

# exclude anchor cities + junk names
junk_names = {"city", "summary", "compare", "output", "outputs", "alphaearth"}
best_new_df = best_df[
    (~best_df["city"].isin(ANCHOR_CITIES)) &
    (~best_df["city"].isin(junk_names))
].copy()

# simple priority
if not best_new_df.empty:
    best_new_df["priority"] = (
        best_new_df["score"].rank(method="dense", ascending=False)
    ).astype(int)
    best_new_df = best_new_df.sort_values(
        ["score", "size_mb", "city"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

suggest_df = best_new_df.head(5).copy()

# =========================
# save outputs
# =========================
all_csv = OUT / "city_candidate_input_files_all.csv"
best_csv = OUT / "city_candidate_input_files_best_per_city.csv"
suggest_csv = OUT / "city_candidate_input_files_suggest_top5.csv"

cand_df.to_csv(all_csv, index=False)
best_df.to_csv(best_csv, index=False)
suggest_df.to_csv(suggest_csv, index=False)

# =========================
# display final tables
# =========================
print("\n===== Best file per city =====")
display(best_df)

print("\n===== New city candidates only =====")
display(best_new_df)

print("\n===== Suggested top candidates =====")
display(suggest_df)

print("\nSaved files:")
print(all_csv)
print(best_csv)
print(suggest_csv)

ROOT: /Users/yufeizhou/Desktop/heat-exposure-compare
DATA_PROCESSED exists: True
CITY_INTAKE exists: True
OUT exists: True

data_processed: found 24 plausible files

city_intake: found 0 plausible files


,city,source_tag,file_path,file_name,suffix,score,size_mb
0,houston,data_processed,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_with_lst_hi_fixed.gpkg,.gpkg,17,1.598
1,houston,data_processed,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_with_lst_hi.gpkg,.gpkg,15,1.594
2,houston,data_processed,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_lst_hi_hotcompare.gpkg,.gpkg,12,1.605
3,houston,data_processed,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_with_lst_hotspot.gpkg,.gpkg,12,1.539
4,houston,data_processed,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_with_lst.gpkg,.gpkg,12,1.527
5,houston,data_processed,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_gistar.gpkg,.gpkg,9,1.582
6,houston,data_processed,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_demo_poverty.gpkg,.gpkg,9,1.496
7,houston,data_processed,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_demo.gpkg,.gpkg,9,1.461
8,houston,data_processed,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_demo_clean.gpkg,.gpkg,9,1.461
9,houston,data_processed,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_tracts_projected.gpkg,.gpkg,7,3.336



===== Best file per city =====


,city,source_tag,file_path,file_name,suffix,score,size_mb
0,houston,data_processed,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_with_lst_hi_fixed.gpkg,.gpkg,17,1.598
1,phoenix,data_processed,/Users/yufeizhou/Desktop/heat-exposure-compare...,phoenix_master_with_lst_hi_fixed.gpkg,.gpkg,17,1.012



===== New city candidates only =====


,city,source_tag,file_path,file_name,suffix,score,size_mb



===== Suggested top candidates =====


,city,source_tag,file_path,file_name,suffix,score,size_mb



Saved files:
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/city_candidate_input_files_all.csv
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/city_candidate_input_files_best_per_city.csv
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/city_candidate_input_files_suggest_top5.csv


In [3]:
from pathlib import Path
import pandas as pd
import re

ROOT = Path("/Users/yufeizhou/Desktop/heat-exposure-compare")

SCAN_ROOTS = [
    ROOT / "data_processed",
    ROOT / "data_raw",
    ROOT / "city_intake",
]

ANCHOR_CITIES = {"houston", "phoenix"}

VALID_EXTS = {".gpkg", ".geojson", ".csv", ".parquet", ".shp", ".json"}

GENERIC_TOKENS = {
    "data", "processed", "raw", "city", "cities", "output", "outputs",
    "pilot", "alphaearth", "compare", "comparison", "summary", "meeting",
    "teacher", "final", "master", "with", "clean", "fixed", "projected",
    "demo", "tract", "tracts", "mosaic", "mean", "scores", "score",
    "corrected", "lst", "hi", "heat", "hotspot", "hotcompare", "gis", "gistar"
}

def clean_token(x):
    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9_]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x

def tokenize_name(name):
    s = clean_token(name)
    toks = [t for t in s.split("_") if t]
    return toks

def plausible_file(p: Path):
    if not p.is_file():
        return False
    if p.suffix.lower() not in VALID_EXTS:
        return False
    s = p.name.lower()
    good_signal = any(k in s for k in [
        "tract", "lst", "hi", "heat", "master", "hotcompare", "hotspot", "mosaic"
    ])
    return good_signal

rows = []

for root in SCAN_ROOTS:
    print(f"\n===== scanning: {root} =====")
    print("exists:", root.exists())
    if not root.exists():
        continue

    # 1) first-level folders
    first_level_dirs = sorted([p for p in root.iterdir() if p.is_dir()])
    dir_rows = []
    for d in first_level_dirs:
        cname = clean_token(d.name)
        dir_rows.append({
            "scan_root": root.name,
            "evidence_type": "folder",
            "candidate_city": cname,
            "path": str(d),
            "file_name": "",
            "score": 5 if cname not in GENERIC_TOKENS else 0
        })

    dir_df = pd.DataFrame(dir_rows)
    if not dir_df.empty:
        print("\nFirst-level folders:")
        display(dir_df)

    # 2) plausible files
    file_rows = []
    for p in root.rglob("*"):
        if not plausible_file(p):
            continue

        parent_city = clean_token(p.parent.name)
        stem_tokens = tokenize_name(p.stem)

        # candidate from parent folder
        if parent_city and parent_city not in GENERIC_TOKENS:
            file_rows.append({
                "scan_root": root.name,
                "evidence_type": "parent_folder",
                "candidate_city": parent_city,
                "path": str(p),
                "file_name": p.name,
                "score": 6
            })

        # candidates from filename tokens
        for tok in stem_tokens:
            if tok in GENERIC_TOKENS:
                continue
            if tok.isdigit():
                continue
            if len(tok) <= 2:
                continue
            file_rows.append({
                "scan_root": root.name,
                "evidence_type": "filename_token",
                "candidate_city": tok,
                "path": str(p),
                "file_name": p.name,
                "score": 2
            })

    file_df = pd.DataFrame(file_rows)

    if not file_df.empty:
        file_df = file_df.drop_duplicates().sort_values(
            ["candidate_city", "score", "file_name"],
            ascending=[True, False, True]
        ).reset_index(drop=True)

        print("\nPlausible file evidence preview:")
        display(file_df.head(50))

    rows.extend(dir_rows)
    rows.extend(file_rows)

# combine all evidence
audit_df = pd.DataFrame(rows)

if audit_df.empty:
    print("\nNo audit evidence found.")
else:
    audit_df["candidate_city"] = audit_df["candidate_city"].map(clean_token)

    # remove generic / anchors
    audit_df = audit_df[
        (~audit_df["candidate_city"].isin(GENERIC_TOKENS)) &
        (~audit_df["candidate_city"].isin(ANCHOR_CITIES)) &
        (audit_df["candidate_city"] != "")
    ].copy()

    # aggregate
    if audit_df.empty:
        print("\nAfter filtering, no non-anchor city candidates were found.")
        summary_df = pd.DataFrame(columns=[
            "candidate_city", "n_evidence_rows", "max_score", "sample_paths"
        ])
    else:
        summary_df = (
            audit_df.groupby("candidate_city", as_index=False)
            .agg(
                n_evidence_rows=("candidate_city", "size"),
                max_score=("score", "max"),
                sample_paths=("path", lambda x: " | ".join(list(pd.Series(x).drop_duplicates().head(3))))
            )
            .sort_values(["max_score", "n_evidence_rows", "candidate_city"], ascending=[False, False, True])
            .reset_index(drop=True)
        )

    audit_csv = ROOT / "outputs" / "pilot" / "alphaearth" / "city_repo_wide_audit.csv"
    summary_csv = ROOT / "outputs" / "pilot" / "alphaearth" / "city_repo_wide_candidate_summary.csv"

    audit_df.to_csv(audit_csv, index=False)
    summary_df.to_csv(summary_csv, index=False)

    print("\n===== Repo-wide candidate summary =====")
    display(summary_df)

    print("\nSaved files:")
    print(audit_csv)
    print(summary_csv)


===== scanning: /Users/yufeizhou/Desktop/heat-exposure-compare/data_processed =====
exists: True

First-level folders:


,scan_root,evidence_type,candidate_city,path,file_name,score
0,data_processed,folder,houston,/Users/yufeizhou/Desktop/heat-exposure-compare...,,5
1,data_processed,folder,phoenix,/Users/yufeizhou/Desktop/heat-exposure-compare...,,5



Plausible file evidence preview:


,scan_root,evidence_type,candidate_city,path,file_name,score
0,data_processed,parent_folder,houston,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_city_tracts.gpkg,6
1,data_processed,parent_folder,houston,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_demo.gpkg,6
2,data_processed,parent_folder,houston,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_demo_clean.gpkg,6
3,data_processed,parent_folder,houston,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_demo_poverty.gpkg,6
4,data_processed,parent_folder,houston,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_gistar.gpkg,6
5,data_processed,parent_folder,houston,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_lst_hi_hotcompare.gpkg,6
6,data_processed,parent_folder,houston,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_with_lst.gpkg,6
7,data_processed,parent_folder,houston,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_with_lst_hi.gpkg,6
8,data_processed,parent_folder,houston,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_with_lst_hi_fixed.gpkg,6
9,data_processed,parent_folder,houston,/Users/yufeizhou/Desktop/heat-exposure-compare...,houston_master_with_lst_hotspot.gpkg,6



===== scanning: /Users/yufeizhou/Desktop/heat-exposure-compare/data_raw =====
exists: True

First-level folders:


...


===== scanning: /Users/yufeizhou/Desktop/heat-exposure-compare/city_intake =====
exists: True

First-level folders:


,scan_root,evidence_type,candidate_city,path,file_name,score
0,city_intake,folder,las_vegas,/Users/yufeizhou/Desktop/heat-exposure-compare...,,5
1,city_intake,folder,miami,/Users/yufeizhou/Desktop/heat-exposure-compare...,,5



===== Repo-wide candidate summary =====


,candidate_city,n_evidence_rows,max_score,sample_paths
0,tract_2025,2,6,/Users/yufeizhou/Desktop/heat-exposure-compare...
1,las_vegas,1,5,/Users/yufeizhou/Desktop/heat-exposure-compare...
2,miami,1,5,/Users/yufeizhou/Desktop/heat-exposure-compare...
3,poverty,2,2,/Users/yufeizhou/Desktop/heat-exposure-compare...



Saved files:
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/city_repo_wide_audit.csv
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/city_repo_wide_candidate_summary.csv


In [5]:
from pathlib import Path
import pandas as pd
import re

ROOT = Path("/Users/yufeizhou/Desktop/heat-exposure-compare")
INTAKE = ROOT / "city_intake"
OUT = ROOT / "outputs" / "pilot" / "alphaearth"
OUT.mkdir(parents=True, exist_ok=True)

TARGET_CITIES = ["las_vegas", "miami"]

VALID_EXTS = {".gpkg", ".geojson", ".csv", ".parquet", ".shp", ".json"}

def clean_name(x):
    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9_]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x

def file_score(name_lower):
    score = 0

    # heat / tract master signals
    if any(k in name_lower for k in ["master", "tract", "tracts"]):
        score += 3
    if any(k in name_lower for k in ["lst", "hi", "heat", "hotspot", "hotcompare"]):
        score += 3

    # embedding signals
    if any(k in name_lower for k in ["alphaearth", "embed", "embedding", "mosaic", "tract_mean"]):
        score += 4

    # pc signals
    if any(k in name_lower for k in ["pc", "pcs", "score", "scores"]):
        score += 4

    # preferred structured formats
    if any(name_lower.endswith(ext) for ext in [".gpkg", ".geojson", ".parquet", ".csv"]):
        score += 1

    return score

def classify_file(name_lower):
    tags = []

    # likely heat table / tract table
    if (
        any(k in name_lower for k in ["master", "tract", "tracts"]) and
        any(k in name_lower for k in ["lst", "hi", "heat", "hotspot", "hotcompare"])
    ):
        tags.append("heat_candidate")

    # broad heat fallback
    elif any(k in name_lower for k in ["lst", "hi", "heat"]):
        tags.append("heat_candidate")

    # alphaearth embedding
    if any(k in name_lower for k in ["alphaearth", "embed", "embedding", "mosaic", "tract_mean"]):
        tags.append("embed_candidate")

    # PC / scores
    if any(k in name_lower for k in ["pc", "pcs", "score", "scores"]):
        tags.append("pc_candidate")

    if not tags:
        tags.append("other")

    return "|".join(tags)

all_rows = []

for city in TARGET_CITIES:
    city_dir = INTAKE / city
    print(f"\n===== {city} =====")
    print("folder exists:", city_dir.exists())
    if not city_dir.exists():
        continue

    file_rows = []
    for p in city_dir.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix.lower() not in VALID_EXTS:
            continue

        name_lower = p.name.lower()
        rel = str(p.relative_to(ROOT))

        row = {
            "city": city,
            "file_name": p.name,
            "suffix": p.suffix.lower(),
            "relative_path": rel,
            "size_mb": round(p.stat().st_size / (1024 * 1024), 3),
            "score": file_score(name_lower),
            "tag": classify_file(name_lower),
        }
        file_rows.append(row)
        all_rows.append(row)

    city_df = pd.DataFrame(file_rows)

    if city_df.empty:
        print("No candidate files found.")
        continue

    city_df = city_df.sort_values(
        ["score", "tag", "file_name"],
        ascending=[False, True, True]
    ).reset_index(drop=True)

    print("\nTop files:")
    display(city_df.head(30))

    print("\nTag counts:")
    display(city_df["tag"].value_counts(dropna=False).rename_axis("tag").reset_index(name="n_files"))

audit_df = pd.DataFrame(all_rows)

if audit_df.empty:
    print("\nNo files found under city_intake candidates.")
else:
    audit_csv = OUT / "city_intake_detailed_audit.csv"
    audit_df.to_csv(audit_csv, index=False)

    # city-level readiness summary
    summary_rows = []
    for city in TARGET_CITIES:
        sub = audit_df[audit_df["city"] == city].copy()
        tags = "|".join(sub["tag"].astype(str).tolist())

        heat_sub = sub[sub["tag"].str.contains("heat_candidate", na=False)].sort_values("score", ascending=False)
        embed_sub = sub[sub["tag"].str.contains("embed_candidate", na=False)].sort_values("score", ascending=False)
        pc_sub = sub[sub["tag"].str.contains("pc_candidate", na=False)].sort_values("score", ascending=False)

        best_heat = heat_sub.iloc[0]["relative_path"] if len(heat_sub) else ""
        best_embed = embed_sub.iloc[0]["relative_path"] if len(embed_sub) else ""
        best_pc = pc_sub.iloc[0]["relative_path"] if len(pc_sub) else ""

        has_heat = len(heat_sub) > 0
        has_embed = len(embed_sub) > 0
        has_pc = len(pc_sub) > 0

        if has_heat and has_embed and has_pc:
            readiness = "ready_direct"
        elif has_heat and has_embed and not has_pc:
            readiness = "ready_if_recompute_pc"
        elif has_heat and not has_embed:
            readiness = "missing_embedding"
        elif (not has_heat) and has_embed:
            readiness = "missing_heat_table"
        else:
            readiness = "not_ready"

        priority = (
            3 * int(has_heat) +
            3 * int(has_embed) +
            2 * int(has_pc) +
            sub["score"].max()
        )

        summary_rows.append({
            "city": city,
            "n_candidate_files": len(sub),
            "has_heat_candidate": has_heat,
            "has_embed_candidate": has_embed,
            "has_pc_candidate": has_pc,
            "best_heat_file": best_heat,
            "best_embed_file": best_embed,
            "best_pc_file": best_pc,
            "readiness": readiness,
            "priority_score": priority,
        })

    summary_df = pd.DataFrame(summary_rows).sort_values(
        ["priority_score", "city"],
        ascending=[False, True]
    ).reset_index(drop=True)

    summary_csv = OUT / "city_intake_candidate_readiness.csv"
    summary_df.to_csv(summary_csv, index=False)

    print("\n===== Candidate readiness summary =====")
    display(summary_df)

    print("\n===== Recommended next city =====")
    display(summary_df.head(1))

    print("\nSaved files:")
    print(audit_csv)
    print(summary_csv)


===== las_vegas =====
folder exists: True
No candidate files found.

===== miami =====
folder exists: True
No candidate files found.

No files found under city_intake candidates.


In [6]:
from pathlib import Path
import pandas as pd

ROOT = Path("/Users/yufeizhou/Desktop/heat-exposure-compare")
INTAKE = ROOT / "city_intake"
OUT = ROOT / "outputs" / "pilot" / "alphaearth"
OUT.mkdir(parents=True, exist_ok=True)

target_cities = ["las_vegas", "miami"]

rows = []

for city in target_cities:
    city_dir = INTAKE / city
    print(f"\n===== {city} =====")
    print("folder exists:", city_dir.exists())

    if not city_dir.exists():
        continue

    items = sorted(city_dir.rglob("*"))
    print("total recursive items:", len(items))

    if len(items) == 0:
        print("-> EMPTY folder")
        continue

    for p in items:
        rows.append({
            "city": city,
            "relative_path": str(p.relative_to(ROOT)),
            "name": p.name,
            "is_file": p.is_file(),
            "is_dir": p.is_dir(),
            "suffix": p.suffix.lower(),
            "size_mb": round(p.stat().st_size / (1024 * 1024), 4) if p.is_file() else None,
        })

tree_df = pd.DataFrame(rows)

tree_csv = OUT / "city_intake_recursive_tree.csv"
if not tree_df.empty:
    tree_df.to_csv(tree_csv, index=False)

    print("\n===== Recursive tree preview =====")
    display(tree_df)

    print("\n===== Files only =====")
    display(tree_df[tree_df["is_file"]].reset_index(drop=True))

    print("\n===== Suffix counts =====")
    suffix_df = (
        tree_df[tree_df["is_file"]]
        .groupby(["city", "suffix"], dropna=False)
        .size()
        .reset_index(name="n")
        .sort_values(["city", "n"], ascending=[True, False])
        .reset_index(drop=True)
    )
    display(suffix_df)

    print("\nSaved:")
    print(tree_csv)
else:
    print("\nNo recursive items found under target city folders.")


===== las_vegas =====
folder exists: True
total recursive items: 3

===== miami =====
folder exists: True
total recursive items: 3

===== Recursive tree preview =====


,city,relative_path,name,is_file,is_dir,suffix,size_mb
0,las_vegas,city_intake/las_vegas/final,final,False,True,,None
1,las_vegas,city_intake/las_vegas/raw,raw,False,True,,None
2,las_vegas,city_intake/las_vegas/working,working,False,True,,None
3,miami,city_intake/miami/final,final,False,True,,None
4,miami,city_intake/miami/raw,raw,False,True,,None
5,miami,city_intake/miami/working,working,False,True,,None



===== Files only =====


,city,relative_path,name,is_file,is_dir,suffix,size_mb



===== Suffix counts =====


,city,suffix,n



Saved:
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/city_intake_recursive_tree.csv


In [7]:
from pathlib import Path
import pandas as pd

ROOT = Path("/Users/yufeizhou/Desktop/heat-exposure-compare")
OUT = ROOT / "outputs" / "pilot" / "alphaearth"
OUT.mkdir(parents=True, exist_ok=True)

target_cities = ["las_vegas", "miami"]

required_items = [
    {
        "required_item": "heat_master_table",
        "description": "tract-level heat table with at least GEOID and heat variables such as LST / HI / hi_minus_lst"
    },
    {
        "required_item": "alphaearth_embedding",
        "description": "tract-level AlphaEarth embedding or tract-mean mosaic table"
    },
    {
        "required_item": "pc_scores",
        "description": "PC1-PC7 scores for the city, or enough embedding data to recompute PCA"
    },
]

rows = []
for city in target_cities:
    for item in required_items:
        rows.append({
            "city": city,
            "required_item": item["required_item"],
            "description": item["description"],
            "status": "missing",
            "notes": ""
        })

checklist_df = pd.DataFrame(rows)
checklist_csv = OUT / "city_intake_missing_checklist.csv"
checklist_df.to_csv(checklist_csv, index=False)

print("===== Missing checklist =====")
display(checklist_df)
print("\nSaved:")
print(checklist_csv)

===== Missing checklist =====


,city,required_item,description,status,notes
0,las_vegas,heat_master_table,tract-level heat table with at least GEOID and...,missing,
1,las_vegas,alphaearth_embedding,tract-level AlphaEarth embedding or tract-mean...,missing,
2,las_vegas,pc_scores,"PC1-PC7 scores for the city, or enough embeddi...",missing,
3,miami,heat_master_table,tract-level heat table with at least GEOID and...,missing,
4,miami,alphaearth_embedding,tract-level AlphaEarth embedding or tract-mean...,missing,
5,miami,pc_scores,"PC1-PC7 scores for the city, or enough embeddi...",missing,



Saved:
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/city_intake_missing_checklist.csv


In [8]:
from pathlib import Path
import pandas as pd

ROOT = Path("/Users/yufeizhou/Desktop/heat-exposure-compare")
OUT = ROOT / "outputs" / "pilot" / "alphaearth"
OUT.mkdir(parents=True, exist_ok=True)

target_cities = ["las_vegas", "miami"]

# 1) priority plan
priority_rows = [
    {
        "city": "las_vegas",
        "priority_rank": 1,
        "recommended_now": True,
        "reason": "Dry-climate extension city; easier first test against Phoenix-style workflow.",
        "next_action": "Prepare this city first."
    },
    {
        "city": "miami",
        "priority_rank": 2,
        "recommended_now": False,
        "reason": "Humid-climate contrast city; better as second extension after one new city runs successfully.",
        "next_action": "Prepare after Las Vegas."
    },
]

priority_df = pd.DataFrame(priority_rows).sort_values("priority_rank").reset_index(drop=True)

priority_csv = OUT / "city_extension_priority_plan.csv"
priority_df.to_csv(priority_csv, index=False)

# 2) manifest template
manifest_rows = []
for city in target_cities:
    city_base = ROOT / "city_intake" / city

    manifest_rows.extend([
        {
            "city": city,
            "slot": "heat_master_table",
            "required": True,
            "subfolder": "raw",
            "example_expected_path": str(city_base / "raw" / f"{city}_heat_master_table.csv"),
            "description": "Tract-level heat table with GEOID and target heat variables (LST / HI / hi_minus_lst if available).",
            "status": "missing"
        },
        {
            "city": city,
            "slot": "alphaearth_embedding",
            "required": True,
            "subfolder": "raw",
            "example_expected_path": str(city_base / "raw" / f"{city}_alphaearth_tract_mean_mosaic.csv"),
            "description": "Tract-level AlphaEarth embedding / tract-mean mosaic table.",
            "status": "missing"
        },
        {
            "city": city,
            "slot": "pc_scores",
            "required": False,
            "subfolder": "working",
            "example_expected_path": str(city_base / "working" / f"{city}_alphaearth_pc1_pc7_scores.csv"),
            "description": "Optional if already computed; otherwise we can recompute from embedding.",
            "status": "missing"
        },
        {
            "city": city,
            "slot": "notes",
            "required": False,
            "subfolder": "final",
            "example_expected_path": str(city_base / "final" / f"{city}_intake_notes.md"),
            "description": "Short note documenting data source, date, CRS, and any preprocessing decisions.",
            "status": "missing"
        },
    ])

manifest_df = pd.DataFrame(manifest_rows)

manifest_csv = OUT / "city_intake_manifest_template.csv"
manifest_df.to_csv(manifest_csv, index=False)

# 3) print summary
print("===== Priority plan =====")
display(priority_df)

print("\n===== Intake manifest template =====")
display(manifest_df)

print("\nSaved files:")
print(priority_csv)
print(manifest_csv)

print("\n===== Recommended next city =====")
display(priority_df.head(1))

===== Priority plan =====


,city,priority_rank,recommended_now,reason,next_action
0,las_vegas,1,True,Dry-climate extension city; easier first test ...,Prepare this city first.
1,miami,2,False,Humid-climate contrast city; better as second ...,Prepare after Las Vegas.



===== Intake manifest template =====


,city,slot,required,subfolder,example_expected_path,description,status
0,las_vegas,heat_master_table,True,raw,/Users/yufeizhou/Desktop/heat-exposure-compare...,Tract-level heat table with GEOID and target h...,missing
1,las_vegas,alphaearth_embedding,True,raw,/Users/yufeizhou/Desktop/heat-exposure-compare...,Tract-level AlphaEarth embedding / tract-mean ...,missing
2,las_vegas,pc_scores,False,working,/Users/yufeizhou/Desktop/heat-exposure-compare...,Optional if already computed; otherwise we can...,missing
3,las_vegas,notes,False,final,/Users/yufeizhou/Desktop/heat-exposure-compare...,"Short note documenting data source, date, CRS,...",missing
4,miami,heat_master_table,True,raw,/Users/yufeizhou/Desktop/heat-exposure-compare...,Tract-level heat table with GEOID and target h...,missing
5,miami,alphaearth_embedding,True,raw,/Users/yufeizhou/Desktop/heat-exposure-compare...,Tract-level AlphaEarth embedding / tract-mean ...,missing
6,miami,pc_scores,False,working,/Users/yufeizhou/Desktop/heat-exposure-compare...,Optional if already computed; otherwise we can...,missing
7,miami,notes,False,final,/Users/yufeizhou/Desktop/heat-exposure-compare...,"Short note documenting data source, date, CRS,...",missing



Saved files:
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/city_extension_priority_plan.csv
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/city_intake_manifest_template.csv

===== Recommended next city =====


,city,priority_rank,recommended_now,reason,next_action
0,las_vegas,1,True,Dry-climate extension city; easier first test ...,Prepare this city first.


In [9]:
from pathlib import Path
import pandas as pd
import re

ROOT = Path("/Users/yufeizhou/Desktop/heat-exposure-compare")
OUT = ROOT / "outputs" / "pilot" / "alphaearth"
OUT.mkdir(parents=True, exist_ok=True)

city = "las_vegas"
city_root = ROOT / "city_intake" / city

print("city_root:", city_root)
print("exists:", city_root.exists())

if not city_root.exists():
    raise FileNotFoundError(f"{city_root} does not exist")

# -----------------------------
# 1) recursive inventory
# -----------------------------
rows = []
for p in sorted(city_root.rglob("*")):
    rel = p.relative_to(ROOT).as_posix()
    rows.append({
        "city": city,
        "relative_path": rel,
        "name": p.name,
        "is_file": p.is_file(),
        "is_dir": p.is_dir(),
        "suffix": p.suffix.lower() if p.is_file() else "",
        "size_mb": round(p.stat().st_size / (1024 * 1024), 4) if p.is_file() else None,
        "parent": p.parent.name,
    })

tree_df = pd.DataFrame(rows)

tree_csv = OUT / f"{city}_intake_recursive_inventory.csv"
tree_df.to_csv(tree_csv, index=False)

print("\n===== Recursive tree preview =====")
display(tree_df.head(50))

files_df = tree_df[tree_df["is_file"]].copy().reset_index(drop=True)

print("\n===== Files only =====")
display(files_df)

# -----------------------------
# 2) score candidate files
# -----------------------------
def score_heat(name: str, rel: str) -> int:
    s = f"{name} {rel}".lower()
    score = 0
    if any(x in s for x in ["heat", "lst", "hi", "heat_index", "hi_minus_lst"]):
        score += 4
    if any(x in s for x in ["master", "tract", "table", "joined", "final"]):
        score += 3
    if any(x in s for x in [".csv", ".gpkg", ".parquet", ".geojson"]):
        score += 2
    if any(x in s for x in ["raw/", "/raw/", "final/", "/final/"]):
        score += 1
    return score

def score_embed(name: str, rel: str) -> int:
    s = f"{name} {rel}".lower()
    score = 0
    if any(x in s for x in ["alphaearth", "embedding", "embed", "mosaic", "tract_mean"]):
        score += 5
    if any(x in s for x in [".csv", ".parquet"]):
        score += 2
    if any(x in s for x in ["raw/", "/raw/", "final/", "/final/"]):
        score += 1
    return score

def score_pc(name: str, rel: str) -> int:
    s = f"{name} {rel}".lower()
    score = 0
    if any(x in s for x in ["pc", "pca", "score", "scores", "principal"]):
        score += 5
    if any(x in s for x in [".csv", ".parquet"]):
        score += 2
    if any(x in s for x in ["working/", "/working/", "final/", "/final/"]):
        score += 1
    return score

cand_df = files_df.copy()
cand_df["heat_score"] = [
    score_heat(n, r) for n, r in zip(cand_df["name"], cand_df["relative_path"])
]
cand_df["embed_score"] = [
    score_embed(n, r) for n, r in zip(cand_df["name"], cand_df["relative_path"])
]
cand_df["pc_score"] = [
    score_pc(n, r) for n, r in zip(cand_df["name"], cand_df["relative_path"])
]

cand_csv = OUT / f"{city}_candidate_files_scored.csv"
cand_df.to_csv(cand_csv, index=False)

# -----------------------------
# 3) top candidates
# -----------------------------
heat_top = cand_df[cand_df["heat_score"] > 0].sort_values(
    ["heat_score", "size_mb", "relative_path"], ascending=[False, False, True]
).reset_index(drop=True)

embed_top = cand_df[cand_df["embed_score"] > 0].sort_values(
    ["embed_score", "size_mb", "relative_path"], ascending=[False, False, True]
).reset_index(drop=True)

pc_top = cand_df[cand_df["pc_score"] > 0].sort_values(
    ["pc_score", "size_mb", "relative_path"], ascending=[False, False, True]
).reset_index(drop=True)

heat_top_csv = OUT / f"{city}_heat_candidates_top.csv"
embed_top_csv = OUT / f"{city}_embed_candidates_top.csv"
pc_top_csv = OUT / f"{city}_pc_candidates_top.csv"

heat_top.to_csv(heat_top_csv, index=False)
embed_top.to_csv(embed_top_csv, index=False)
pc_top.to_csv(pc_top_csv, index=False)

print("\n===== Top heat candidates =====")
display(heat_top.head(15))

print("\n===== Top embedding candidates =====")
display(embed_top.head(15))

print("\n===== Top PC candidates =====")
display(pc_top.head(15))

# -----------------------------
# 4) compact recommendation
# -----------------------------
pick_rows = []
pick_rows.append({
    "city": city,
    "slot": "heat_master_table",
    "recommended_path": heat_top.iloc[0]["relative_path"] if len(heat_top) else None,
    "score": heat_top.iloc[0]["heat_score"] if len(heat_top) else None,
})
pick_rows.append({
    "city": city,
    "slot": "alphaearth_embedding",
    "recommended_path": embed_top.iloc[0]["relative_path"] if len(embed_top) else None,
    "score": embed_top.iloc[0]["embed_score"] if len(embed_top) else None,
})
pick_rows.append({
    "city": city,
    "slot": "pc_scores",
    "recommended_path": pc_top.iloc[0]["relative_path"] if len(pc_top) else None,
    "score": pc_top.iloc[0]["pc_score"] if len(pc_top) else None,
})

pick_df = pd.DataFrame(pick_rows)
pick_csv = OUT / f"{city}_recommended_input_slots.csv"
pick_df.to_csv(pick_csv, index=False)

print("\n===== Recommended input slots =====")
display(pick_df)

print("\nSaved files:")
print(tree_csv)
print(cand_csv)
print(heat_top_csv)
print(embed_top_csv)
print(pc_top_csv)
print(pick_csv)

city_root: /Users/yufeizhou/Desktop/heat-exposure-compare/city_intake/las_vegas
exists: True

===== Recursive tree preview =====


,city,relative_path,name,is_file,is_dir,suffix,size_mb,parent
0,las_vegas,city_intake/las_vegas/final,final,False,True,,None,las_vegas
1,las_vegas,city_intake/las_vegas/raw,raw,False,True,,None,las_vegas
2,las_vegas,city_intake/las_vegas/working,working,False,True,,None,las_vegas



===== Files only =====


,city,relative_path,name,is_file,is_dir,suffix,size_mb,parent



===== Top heat candidates =====


...


===== Top PC candidates =====


,city,relative_path,name,is_file,is_dir,suffix,size_mb,parent,heat_score,embed_score,pc_score



===== Recommended input slots =====


,city,slot,recommended_path,score
0,las_vegas,heat_master_table,None,None
1,las_vegas,alphaearth_embedding,None,None
2,las_vegas,pc_scores,None,None



Saved files:
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/las_vegas_intake_recursive_inventory.csv
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/las_vegas_candidate_files_scored.csv
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/las_vegas_heat_candidates_top.csv
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/las_vegas_embed_candidates_top.csv
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/las_vegas_pc_candidates_top.csv
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/las_vegas_recommended_input_slots.csv


### Interim conclusion for Las Vegas intake

I checked the local project folders and broader Desktop/Documents/Downloads paths.

Result:
- `city_intake/las_vegas/` exists, but only contains empty `raw`, `working`, and `final` folders.
- No valid tract-level heat input file was found locally for Las Vegas.
- No valid AlphaEarth embedding file was found locally for Las Vegas.
- No valid PC score file was found locally for Las Vegas.
- The files detected in this step were only the audit outputs generated by the notebook itself, not usable city inputs.

Conclusion:
Las Vegas is still the preferred next city conceptually, but it is **not yet runnable locally** until the required source files are added into `city_intake/las_vegas/`.

In [10]:
import pandas as pd
from pathlib import Path

OUT = Path("/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth")

final_status = pd.DataFrame([
    {
        "city": "las_vegas",
        "heat_master_table_found": False,
        "alphaearth_embedding_found": False,
        "pc_scores_found": False,
        "city_folder_exists": True,
        "ready_to_run": False,
        "conclusion": "Preferred next city conceptually, but not runnable yet because required source files are missing locally."
    },
    {
        "city": "miami",
        "heat_master_table_found": False,
        "alphaearth_embedding_found": False,
        "pc_scores_found": False,
        "city_folder_exists": True,
        "ready_to_run": False,
        "conclusion": "Backup candidate only; also not runnable yet because required source files are missing locally."
    }
])

save_path = OUT / "city_extension_readiness_final.csv"
final_status.to_csv(save_path, index=False)

print(save_path)
display(final_status)

/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/alphaearth/city_extension_readiness_final.csv


,city,heat_master_table_found,alphaearth_embedding_found,pc_scores_found,city_folder_exists,ready_to_run,conclusion
0,las_vegas,False,False,False,True,False,"Preferred next city conceptually, but not runn..."
1,miami,False,False,False,True,False,Backup candidate only; also not runnable yet b...
